# Tine: a larger open model as the verifier

Use an **A100 GPU** runtime (40 GB). The model is about 20 GB; an L4 or T4 is too small.
Run the cells in order. About 10 minutes of downloads, then roughly 20-40 minutes of compute.

What it does: runs `qwen2.5-coder:32b` with Ollama on the Colab GPU and asks it, for each of the 150
pilot patches, whether the patch resolves its issue. It is the same script, prompt and runner as the
14B run on the Mac (`results/2026-11-11-judge-pilot`), so the only thing that changes is the model's
size. The model only reads the patches. **No patch is executed.** See `docs/JUDGE.md`.

Needs `cases-pilot.jsonl` (from `artifacts/judge/` on the Mac). Put it in `MyDrive/tine/`, or cell 1
asks you to upload it. It holds issue text and model patches, so it stays on Drive and is never
committed. The grades are not needed here: scoring happens afterwards from the verdict files.

In [ ]:
# 1. Mount Drive, get the repository, check the GPU and the cases.
from google.colab import drive
drive.mount("/content/drive")

import json, os, shutil, subprocess, sys, time, urllib.request
from pathlib import Path

BRANCH = "claude/colab-judge"  # falls back to "main" once this branch is merged and deleted
ORIGIN = "https://github.com/Nandu-Dammalapti/tine.git"
if subprocess.run(["git", "ls-remote", "--exit-code", "--heads", ORIGIN, BRANCH], capture_output=True).returncode:
    BRANCH = "main"
MODEL = "qwen2.5-coder:32b"
ROOT = Path("/content/drive/MyDrive/tine")
REPO = Path("/content/tine")
if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "fetch", "-q", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "-q", "-B", BRANCH, "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", BRANCH,
                    ORIGIN, str(REPO)], check=True)
os.chdir(REPO)
REVISION = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("tine revision:", REVISION)
GPU = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print(GPU or "NO GPU: change the runtime type")

CASES = ROOT / "cases-pilot.jsonl"
if not CASES.exists():
    from google.colab import files
    print("Upload cases-pilot.jsonl (artifacts/judge/cases-pilot.jsonl on the Mac)")
    uploaded = files.upload()
    ROOT.mkdir(parents=True, exist_ok=True)
    CASES.write_bytes(uploaded["cases-pilot.jsonl"])
count = sum(1 for line in CASES.open(encoding="utf-8") if line.strip())
print("cases:", count)
assert count == 150, "expected the 150 pilot cases"

In [ ]:
# 2. Install Ollama, start it, download the model (about 20 GB) and load it onto the GPU.
subprocess.run("apt-get install -y -qq zstd pciutils > /dev/null", shell=True, check=True)
if not shutil.which("ollama"):
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)

URL = "http://127.0.0.1:11434"
def alive():
    try:
        return urllib.request.urlopen(URL + "/api/version", timeout=3).status == 200
    except OSError:
        return False

if not alive():
    # A 16k context so that no issue or patch is silently cut off.
    env = dict(os.environ, OLLAMA_CONTEXT_LENGTH="16384", OLLAMA_KEEP_ALIVE="2h")
    subprocess.Popen(["ollama", "serve"], env=env, stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(60):
        if alive():
            break
        time.sleep(1)
assert alive(), "Ollama did not start: see /content/ollama.log"
subprocess.run(["ollama", "pull", MODEL], check=True)
subprocess.run(["ollama", "run", MODEL, "Reply with the word ready."], check=True)
print(subprocess.run(["ollama", "ps"], capture_output=True, text=True).stdout)
print("The PROCESSOR column must say 100% GPU. If it says CPU, stop: the runtime is too small.")

In [ ]:
# 3. Smoke test on 5 cases.
def run_judge(cases, output, style, max_tokens):
    name = MODEL if style == "verdict" else f"{MODEL} ({style})"
    subprocess.run([sys.executable, "scripts/judge_patches.py", "judge", "--cases", str(cases),
                    "--output", str(output), "--base-url", URL + "/v1", "--model", MODEL,
                    "--judge-name", name, "--style", style, "--max-tokens", str(max_tokens)], check=True)

smoke_cases = Path("/content/cases-smoke.jsonl")
smoke_cases.write_text("".join(CASES.open(encoding="utf-8").readlines()[:5]), encoding="utf-8")
smoke_out = Path("/content/verdicts-smoke.csv")
smoke_out.unlink(missing_ok=True)
run_judge(smoke_cases, smoke_out, "verdict", 400)
print(smoke_out.read_text())

In [ ]:
# 4. All 150 cases, in both styles. Resumable: after a disconnect rerun cells 1, 2 and 4.
slug = MODEL.replace(":", "-").replace(".", "")
VERDICT = ROOT / f"judge-verdicts-{slug}.csv"
THINK = ROOT / f"judge-verdicts-{slug}-think.csv"
run_judge(CASES, VERDICT, "verdict", 400)   # bare verdict
run_judge(CASES, THINK, "think", 900)       # reason through the change first

In [ ]:
# 5. Summary and run record. Copy this output back to Claude.
import csv
from collections import Counter
listing = subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout
version = subprocess.run(["ollama", "--version"], capture_output=True, text=True).stdout.strip()
record = {"model": MODEL, "ollama_list": listing, "ollama_version": version, "context_length": 16384,
          "gpu": GPU, "tine_revision": REVISION, "cases": str(CASES.name)}
for path in (VERDICT, THINK):
    rows = list(csv.DictReader(path.open(encoding="utf-8")))
    seconds = [float(r["seconds"]) for r in rows if r["seconds"]]
    prompt = [int(r["prompt_tokens"]) for r in rows if r["prompt_tokens"]]
    summary = {"rows": len(rows), "verdicts": dict(Counter(r["verdict"] for r in rows)),
               "mean_seconds": round(sum(seconds) / len(seconds), 1), "max_prompt_tokens": max(prompt)}
    record[path.name] = summary
    print(path.name, summary)
(ROOT / f"judge-run-{slug}.json").write_text(json.dumps(record, indent=2))
print(listing)
print("saved:", VERDICT, THINK, sep="\n")

Done. Choose **Runtime → Disconnect and delete runtime**. The verdicts are on Drive in `MyDrive/tine/`
as `judge-verdicts-qwen25-coder-32b.csv` and `judge-verdicts-qwen25-coder-32b-think.csv`. Scoring
against the grades (`scripts/judge_patches.py score`) runs from those files on any machine that has
the key file.